# Notebook 3 — Per-Stage Steady-State Analysis

This notebook performs a detailed steady-state analysis of each DR stage:
- Heat balance at the nominal operating point
- Sensitivity to parameter variations
- Stage coupling analysis (how each stage affects its neighbors)

## Heat balance equation at steady state

At steady state $\dot{T}_i = 0$:

$$G_i (T_{i-1} - T_i^*) - G_{i+1}(T_i^* - T_{i+1}^*) = P_{\text{cool},i}(T_i^*) - P_{\text{qubit},i}$$

For the MXC (stage 6, last stage):
$$G_6 (T_5^* - T_6^*) + P_{\text{qubit}} = P_{\text{cool},6}(T_6^*) = 0.04\, (T_6^*)^2$$

In [ ]:
import sys
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from cryo_thermal import simulate_cooldown, heat_balance, STAGE_NAMES, _G0, _C

# Run baseline simulation
t_h, T = simulate_cooldown(t_hours=10.0, n_eval=2000)
T_ss = T[-1]

print('Steady-state temperatures:')
for i, (nm, t) in enumerate(zip(STAGE_NAMES, T_ss)):
    if t > 1:
        print(f'  {nm:22s}: {t:.3f} K')
    else:
        print(f'  {nm:22s}: {t*1e3:.3f} mK')

In [ ]:
# ── Heat balance table ───────────────────────────────────────────────────────
balance = heat_balance(T_ss)

print(f"\n{'Stage':<22} {'P_cool':>12} {'Q_in (net)':>13} {'|Balance|':>12}")
print('─' * 62)
for row in balance:
    P = row['P_cool_W']
    Q = row['Q_in_W']
    b = abs(row['balance_W'])
    # Auto-scale units
    def fmt_W(v):
        if abs(v) >= 0.01: return f'{v*1e3:.3f} mW'
        if abs(v) >= 1e-5: return f'{v*1e6:.2f} µW'
        return f'{v*1e9:.2f} nW'
    print(f"{row['stage']:<22} {fmt_W(P):>12} {fmt_W(Q):>13} {fmt_W(b):>12}")

In [ ]:
# ── Heat balance bar chart ────────────────────────────────────────────────────
stages_sub = ['50K Stage', '4K Stage', 'Still', 'Cold Plate', 'MXC']
P_cool_vals = [row['P_cool_W'] for row in balance]
Q_in_vals   = [row['Q_in_W']  for row in balance]

# Convert to µW for visibility
P_cool_uW = [v * 1e6 for v in P_cool_vals]
Q_in_uW   = [v * 1e6 for v in Q_in_vals]

x = np.arange(len(stages_sub))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - w/2, P_cool_uW, w, label='P_cool (available)', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + w/2, Q_in_uW,   w, label='Q_in (heat load)',   color='tomato',    alpha=0.85)
ax.set_yscale('log')
ax.set_xticks(x); ax.set_xticklabels(stages_sub, fontsize=11)
ax.set_ylabel('Power (µW)', fontsize=12)
ax.set_title('DR Heat Balance at Steady State (Log Scale)', fontsize=13)
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../outputs/heat_balance_stages.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Conductance sensitivity sweep ────────────────────────────────────────────
# How does MXC temperature change if inter-stage conductances increase by ±50%?

scale_factors = np.linspace(0.5, 2.5, 25)
T_mxc_g45 = []
T_mxc_g56 = []

for sf in scale_factors:
    # Scale G45 (Still→CP)
    from cryo_thermal import _G0 as G_base
    G_mod = G_base.copy(); G_mod[4] = G_base[4] * sf
    _, T_mod = simulate_cooldown(n_eval=800, params={'G_override': G_mod})
    T_mxc_g45.append(T_mod[-1, 5] * 1e3)

    # Scale G56 (CP→MXC)
    G_mod = G_base.copy(); G_mod[5] = G_base[5] * sf
    _, T_mod = simulate_cooldown(n_eval=800, params={'G_override': G_mod})
    T_mxc_g56.append(T_mod[-1, 5] * 1e3)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(scale_factors, T_mxc_g45, 'b-o', ms=5, lw=2, label='G₄₅ (Still→CP) scaled')
ax.plot(scale_factors, T_mxc_g56, 'r-s', ms=5, lw=2, label='G₅₆ (CP→MXC) scaled')
ax.axhline(20, color='k', ls='--', lw=1.5, label='20 mK limit')
ax.axvline(1.0, color='gray', ls=':', alpha=0.7, label='Nominal')
ax.set_xlabel('Conductance scale factor', fontsize=12)
ax.set_ylabel('MXC temperature (mK)', fontsize=12)
ax.set_title('MXC Temperature vs. Sub-K Conductance', fontsize=12)
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/conductance_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Stage coupling: what happens if 4K stage warms up? ───────────────────────
# Simulate with elevated 4K stage (e.g., due to poor PT2 vacuum or high load)

from cryo_thermal import _G0 as G_base

# Reduce 4K PT cooling power by varying the base temperature offset
pt2_loads = np.linspace(0, 0.5, 20)   # extra static load on 4K stage [W]
T_4K_ss  = []
T_mxc_ss = []

for extra in pt2_loads:
    _, T_mod = simulate_cooldown(n_eval=800, params={'P_extra_4K': extra})
    T_4K_ss.append(T_mod[-1, 2])
    T_mxc_ss.append(T_mod[-1, 5] * 1e3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(pt2_loads * 1e3, T_4K_ss, 'b-o', ms=5, lw=2)
axes[0].set_xlabel('Extra 4K load (mW)', fontsize=12)
axes[0].set_ylabel('4K stage temperature (K)', fontsize=12)
axes[0].set_title('4K Stage T vs. Extra Load', fontsize=11)
axes[0].grid(alpha=0.3)

axes[1].plot(pt2_loads * 1e3, T_mxc_ss, 'r-s', ms=5, lw=2)
axes[1].axhline(20, color='k', ls='--', lw=1.5, label='20 mK limit')
axes[1].set_xlabel('Extra 4K load (mW)', fontsize=12)
axes[1].set_ylabel('MXC temperature (mK)', fontsize=12)
axes[1].set_title('MXC T vs. Extra 4K Load (Stage Coupling)', fontsize=11)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/stage_coupling.png', dpi=150, bbox_inches='tight')
plt.show()